## 今天使用的 4 張資料表

### customers
一列 = 一位客戶

- `customer_id`：客戶編號
- `city`：城市
- `segment`：客群
- `acquisition_channel`：獲客管道

### orders
一列 = 一張訂單

- `order_id`：訂單編號
- `customer_id`：客戶編號
- `order_date`：訂單日期
- `status`：訂單狀態
- `payment_type`：付款方式

### order_items
一列 = 一個商品明細

- `order_id`：訂單編號
- `product_id`：商品編號
- `quantity`：數量
- `unit_price`：單價
- `discount_rate`：折扣率

### products
一列 = 一個商品

- `product_id`：商品編號
- `product_name`：商品名稱
- `category`：商品類別

In [ ]:
import sqlite3  # 匯入 Python 內建的 SQLite 模組
import pandas as pd  # 匯入 pandas，SQL 查詢結果會轉成 DataFrame
conn = sqlite3.connect("course.db")
sql = "SELECT * FROM sqlite_master WHERE type='table';"
pd.read_sql_query(sql, conn)


,type,name,tbl_name,rootpage,sql
0,table,customers,customers,2,CREATE TABLE customers (\n customer_id TEXT...
1,table,orders,orders,4,CREATE TABLE orders (\n order_id TEXT PRIMA...
2,table,products,products,6,CREATE TABLE products (\n product_id TEXT P...
3,table,order_items,order_items,8,"CREATE TABLE order_items (\n order_id TEXT,..."


In [5]:
sql = '''
SELECT
    customer_id,  -- 客戶編號
    city,         -- 城市
    segment       -- 客群
FROM customers
LIMIT 10;
'''
pd.read_sql_query(sql, conn)

,customer_id,city,segment
0,C001,Taipei,new
1,C002,Kaohsiung,growth
2,C003,Taichung,vip
3,C004,Tainan,new
4,C005,Taipei,growth
5,C006,Kaohsiung,vip
6,C007,Taichung,new
7,C008,Tainan,growth
8,C009,Taipei,vip
9,C010,Kaohsiung,new


In [8]:
sql = '''
SELECT
    *
FROM customers
WHERE segment = 'vip'
'''
pd.read_sql_query(sql, conn)

,customer_id,city,segment,acquisition_channel
0,C003,Taichung,vip,referral
1,C006,Kaohsiung,vip,ads
2,C009,Taipei,vip,organic
3,C012,Tainan,vip,email
4,C015,Taichung,vip,referral
5,C018,Kaohsiung,vip,ads
6,C021,Taipei,vip,organic
7,C024,Tainan,vip,email
8,C027,Taichung,vip,referral
9,C030,Kaohsiung,vip,ads


## AND、OR、IN

### AND
兩個條件都要成立。

### OR
符合其中一個條件即可。

### IN
同一個欄位要比對很多選項時，比很多個 `OR` 更好讀。


In [17]:
# 找出「高雄而且是 VIP」的客戶
city = "Tainan"
sql = f"""
SELECT *
FROM customers
WHERE city = '{city}'  -- 第一個條件：城市是高雄
  AND segment = 'vip';    -- 第二個條件：客群是 VIP
"""
pd.read_sql_query(sql, conn)

,customer_id,city,segment,acquisition_channel
0,C012,Tainan,vip,email
1,C024,Tainan,vip,email


In [12]:
sql = """
SELECT *
FROM customers
"""
df = pd.read_sql_query(sql, conn)
df["city"].value_counts()


city
Taipei       8
Kaohsiung    8
Taichung     7
Tainan       7
Name: count, dtype: int64

In [ ]:
sql = """
SELECT *
FROM customers
WHERE city IN (
    'Taipei',     -- 台北
    'Kaohsiung'   -- 高雄
);
"""
pd.read_sql_query(sql, conn)

,customer_id,city,segment,acquisition_channel
0,C001,Taipei,new,organic
1,C002,Kaohsiung,growth,ads
2,C005,Taipei,growth,organic
3,C006,Kaohsiung,vip,ads
4,C009,Taipei,vip,organic
5,C010,Kaohsiung,new,ads
6,C013,Taipei,new,organic
7,C014,Kaohsiung,growth,ads
8,C017,Taipei,growth,organic
9,C018,Kaohsiung,vip,ads


In [23]:
sql = """
SELECT
    order_id,    -- 訂單編號
    product_id,  -- 商品編號
    unit_price   -- 商品單價
FROM order_items
ORDER BY unit_price DESC  -- 依照商品單價由高到低排序
LIMIT 10;  -- 只取前 10 筆
"""
pd.read_sql_query(sql, conn)

,order_id,product_id,unit_price
0,O002,P003,3990.0
1,O003,P003,3990.0
2,O010,P003,3990.0
3,O011,P003,3990.0
4,O017,P003,3990.0
5,O019,P003,3990.0
6,O026,P003,3990.0
7,O027,P003,3990.0
8,O034,P003,3990.0
9,O035,P003,3990.0


常用聚合函數：

| SQL | 白話意思 |
|---|---|
| `COUNT()` | 算有幾筆 |
| `SUM()` | 加總 |
| `AVG()` | 平均 |
| `MIN()` | 最小值 |
| `MAX()` | 最大值 |


In [29]:
sql = """SELECT COUNT(*) AS order_count FROM orders; """
pd.read_sql_query(sql, conn)

,order_count
0,60


In [38]:
sql = """SELECT SUM(quantity) as total_quantity,
                ROUND(AVG(unit_price),2)as avg_quantity
                FROM order_items;"""
pd.read_sql_query(sql, conn)

,total_quantity,avg_quantity
0,136,1457.83


In [51]:
# GROUP BY：分組統計
## 這和 pandas 的 `groupby()` 是同一個概念。

In [46]:
sql = """
SELECT
    payment_type,               -- 付款方式
    COUNT(*) AS order_count     -- 每一組的訂單數
FROM orders
GROUP BY payment_type  -- 依照付款方式分組
ORDER BY order_count DESC;  -- 依照訂單數由高到低排序
"""
pd.read_sql_query(sql, conn)

,payment_type,order_count
0,wallet,15
1,transfer,15
2,cash,15
3,card,15


## WHERE + GROUP BY

商業問題：

> 各付款方式有幾張「已完成」訂單？

In [55]:
sql = """
SELECT
    payment_type,               -- 付款方式
    COUNT(*) AS order_count     -- 已完成訂單數
FROM orders
WHERE status = 'completed'-- 只統計已完成的訂單
GROUP BY payment_type  -- 依照付款方式分組
ORDER BY order_count DESC;  -- 依照訂單數由高到低排序
"""
pd.read_sql_query(sql, conn)

,payment_type,order_count
0,wallet,12
1,transfer,12
2,cash,12
3,card,12


In [ ]:
sql = """
    SELECT COUNT(*) AS order_count,  -- 訂單數
           COUNT(DISTINCT customer_id) AS customer_count  --移除重複客戶數
    FROM orders;
"""
pd.read_sql_query(sql, conn)

,order_count,customer_count
0,60,25


## HAVING

先記這個差別：

```text
WHERE
→ 篩選原始資料

HAVING
→ 篩選 GROUP BY 算完之後的結果
```

In [ ]:
sql = """
SELECT
    payment_type,               -- 付款方式
    COUNT(*) AS order_count     -- 每組訂單數
FROM orders
GROUP BY payment_type           -- 先依付款方式分組
HAVING COUNT(*) >= 10           -- 再篩選分組後的結果
ORDER BY order_count DESC;   
"""

# 執行查詢
pd.read_sql_query(sql, conn)

,payment_type,order_count
0,wallet,15
1,transfer,15
2,cash,15
3,card,15


In [69]:
## INNER JOIN 兩邊都有配對，才留下
sql = """
    SELECT o.order_id,
          o.customer_id,
          c.segment,
          c.city
    FROM orders AS o
    INNER JOIN customers AS c
    ON o.customer_id = c.customer_id
    LIMIT 10;
"""
pd.read_sql_query(sql, conn)

,order_id,customer_id,segment,city
0,O001,C001,new,Taipei
1,O002,C002,growth,Kaohsiung
2,O003,C003,vip,Taichung
3,O004,C004,new,Tainan
4,O005,C005,growth,Taipei
5,O006,C006,vip,Kaohsiung
6,O007,C007,new,Taichung
7,O008,C008,growth,Tainan
8,O009,C009,vip,Taipei
9,O010,C010,new,Kaohsiung


In [ ]:
sql = """
    SELECT c.customer_id,
           c.segment,
           o.order_id
    FROM customers AS c  
    LEFT JOIN orders AS o       
    ON c.customer_id = o.customer_id
    ORDER BY c.customer_id
"""
pd.read_sql_query(sql, conn)


,customer_id,segment,order_id
0,C001,new,O001
1,C001,new,O026
2,C001,new,O051
3,C002,growth,O002
4,C002,growth,O027
...,...,...,...
60,C026,growth,NaN
61,C027,vip,NaN
62,C028,new,NaN
63,C029,growth,NaN



商業問題：

> 哪一個客群的訂單最多？

In [76]:
sql = """
SELECT
    c.segment,  -- 客群
    COUNT(o.order_id) AS order_count  -- 訂單數
    FROM orders AS o
    INNER JOIN customers AS c
    ON o.customer_id = c.customer_id
    GROUP BY c.segment  -- 依照客群分組
    ORDER BY order_count DESC;  -- 依照訂單數由高到低排序
"""
pd.read_sql_query(sql, conn)

,segment,order_count
0,new,22
1,vip,19
2,growth,19


In [80]:
## 哪一個商品類別賣出的商品數量最多？

In [81]:
sql = """ 
SELECT
    p.category,  -- 商品類別
    SUM(oi.quantity) AS total_quantity  -- 該類別賣出的總數量
FROM order_items AS oi
INNER JOIN products AS p
ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY total_quantity DESC;-- 銷售數量多的排前面
"""
pd.read_sql_query(sql, conn)

,category,total_quantity
0,Accessories,53
1,Electronics,51
2,Office,32


營收公式

本堂統一使用：

```text
品項營收
=
quantity × unit_price × (1 - discount_rate)
```

例如：

```text
quantity = 2
unit_price = 1000
discount_rate = 0.1

實付比例 = 1 - 0.1 = 0.9

品項營收
= 2 × 1000 × 0.9
= 1800
```

商業分析時，營收只計算：

```text
status = 'completed'
```

In [83]:
sql = """
SELECT
    order_id,                                      -- 訂單編號
    product_id,                                    -- 商品編號
    quantity,                                      -- 數量
    unit_price,                                    -- 單價
    discount_rate,                                 -- 折扣率
    ROUND(
        quantity * unit_price * (1 - discount_rate),
        2
    ) AS line_revenue                              -- 折扣後的商品明細營收
FROM order_items
LIMIT 10;
"""

# 執行查詢
pd.read_sql_query(sql, conn)

,order_id,product_id,quantity,unit_price,discount_rate,line_revenue
0,O001,P001,1,590.0,0.0,590.0
1,O001,P002,1,1890.0,0.0,1890.0
2,O002,P002,1,1890.0,0.0,1890.0
3,O002,P003,1,3990.0,0.0,3990.0
4,O002,P004,1,990.0,0.0,990.0
5,O003,P003,1,3990.0,0.0,3990.0
6,O004,P004,1,990.0,0.0,990.0
7,O004,P005,1,1290.0,0.1,1161.0
8,O005,P005,1,1290.0,0.1,1161.0
9,O005,P006,1,790.0,0.0,790.0


綜合商業分析：各客群的營收表現

現在把前面的知識串起來。

商業問題：

> 哪一個客群的營收最高？平均每張訂單金額是多少？

需要三張表：

```text
customers
↓
orders
↓
order_items
```

思考順序：

```text
先把訂單與商品明細接起來
↓
只留下 completed
↓
計算每張訂單的營收
↓
再接 customers
↓
依 segment 分組
↓
算訂單數、總營收、平均訂單金額
```

In [84]:
sql = """
WITH order_revenue AS (

    -- 第一步：計算每張已完成訂單的營收
    SELECT
        o.order_id,                                  -- 訂單編號
        o.customer_id,                               -- 客戶編號
        SUM(
            oi.quantity
            * oi.unit_price
            * (1 - oi.discount_rate)
        ) AS revenue                                 -- 每張訂單的總營收

    FROM orders AS o
    INNER JOIN order_items AS oi
        ON o.order_id = oi.order_id                  -- 訂單接商品明細

    WHERE o.status = 'completed'                     -- 只計算已完成訂單

    GROUP BY
        o.order_id,
        o.customer_id                                -- 一列整理成一張訂單
)

-- 第二步：再依客群做商業分析
SELECT
    c.segment,                                       -- 客群
    COUNT(r.order_id) AS completed_orders,           -- 完成訂單數
    ROUND(SUM(r.revenue), 2) AS total_revenue,       -- 客群總營收
    ROUND(AVG(r.revenue), 2) AS aov                  -- 平均每張訂單金額
FROM order_revenue AS r
INNER JOIN customers AS c
    ON r.customer_id = c.customer_id                 -- 接上客群資料
GROUP BY c.segment                                   -- 依客群分組
ORDER BY total_revenue DESC;                         -- 總營收高的排前面
"""
# 執行查詢
segment_result = pd.read_sql_query(sql, conn)

# 顯示客群營收分析結果
segment_result


,segment,completed_orders,total_revenue,aov
0,new,19,69339.5,3649.45
1,growth,15,48906.0,3260.40
2,vip,14,39083.5,2791.68


# 14. EDA 方法論：從問題到假設，再用 SQL 驗證

EDA 是：

> **Exploratory Data Analysis（探索性資料分析）**

但 EDA 不是：

> 「拿到資料後，什麼都查一遍。」

比較好的做法是：

```text
Question（問題）
↓
Hypothesis（假設）
↓
Metric（指標）
↓
SQL 取資料與計算
↓
Table / Chart（表格或圖表）
↓
Insight（洞察）
↓
Action（行動）
```

這和前面學 SQL 的關係非常直接：

```text
SQL 是工具
EDA 是思考方法
```

也就是：

> **先想清楚要回答什麼，再決定 SQL 要怎麼寫。**

---

## 14.1 第一步：Question（問題）

先把模糊問題改成可以分析的問題。

不好的問題：

> 哪個客群比較好？

這句話太模糊，因為「好」可能代表：

- 訂單數比較多
- 營收比較高
- 平均訂單金額比較高
- 回購比較多

改成：

> **哪一個客群的平均訂單金額（AOV）最高？**

這樣才知道接下來要算什麼。

---

## 14.2 第二步：Hypothesis（假設）

在查資料前，先寫下你目前的猜測。

例如：

> **假設：VIP 客群的 AOV 最高。**

注意：

> 假設不是答案，也不是一定要證明它是對的。

如果資料最後不支持這個假設，一樣是很有價值的結果。

---

## 14.3 第三步：Metric（指標）

這題要看的指標是：

```text
AOV
=
完成訂單營收
÷
完成訂單數
```

前面已經算過 `segment_result`，裡面有：

```text
segment
completed_orders
total_revenue
aov
```

所以可以直接拿來驗證假設。


第四步到第七步：從結果走到商業判斷

拿到 SQL 結果後，不要停在：

> 「我查出一張表了。」

要繼續問：

### Result（結果）

先客觀描述：

```text
哪個客群 AOV 最高？
VIP 是不是最高？
差距大約多少？
```

### Insight（洞察）

再解釋這個結果代表什麼。

例如，如果 VIP 並不是 AOV 最高：

> **目前資料沒有支持「VIP 單筆消費一定比較高」這個想法。**

注意，不要直接跳到：

> 「VIP 沒價值。」

因為 AOV 只代表「平均每張訂單金額」，還沒有看：

- 下單次數
- 每位客戶總消費
- 回購狀況

### Action（行動）

比較合理的下一步是：

```text
先確認 VIP 的定義
↓
再看每位客戶的總消費
↓
再看下單次數
↓
最後才決定要不要調整客群策略
```

---
